# Phase 3 - Notebook 00: DUSt3R Overview & Motivation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChunLI-666/3DGS-from-scratch/blob/develop/notebooks/phase3/00_phase3_overview.ipynb)

---

## Learning Objectives

By the end of this notebook, you will:
1. Understand why traditional SfM/SLAM has limitations
2. Know DUSt3R's end-to-end geometric learning approach
3. Understand the Pointmap representation (core innovation)
4. Compare DUSt3R vs COLMAP performance
5. See the evolution from pairwise to multi-view methods
6. Understand DUSt3R's connection to SLAM frontend
7. Have a clear roadmap for Phase 3

**Estimated Time**: 50 minutes

**Prerequisites**: Basic understanding of SLAM, camera geometry, and deep learning

---

## 0. Environment Setup

In [ ]:
# 环境设置 (Environment setup)
import os
import sys

# Colab兼容性检查
if 'COLAB_GPU' in os.environ:
    !pip install -q plotly ipywidgets
    !git clone https://github.com/ChunLI-666/3DGS-from-scratch.git
    %cd 3DGS-from-scratch

# 添加项目根目录到路径
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle, Circle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 中文支持
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成!")
print(f"NumPy version: {np.__version__}")

## 1. Why DUSt3R? Traditional SfM Limitations

### 1.1 传统SfM Pipeline的问题

传统的Structure from Motion (SfM) pipeline (如COLMAP) 采用多阶段处理方式:

```
传统SfM Pipeline (COLMAP):
┌─────────────────────────────────────────────────────────────────┐
│  1. 特征提取 (SIFT/SuperPoint)                                   │
│  2. 特征匹配 + RANSAC几何验证                                    │
│  3. 增量式重建 (三角化 + BA)                                     │
│  4. 稠密重建 (PatchMatch MVS)                                    │
└─────────────────────────────────────────────────────────────────┘
```

**主要问题 (Problems):**
- 多阶段pipeline，误差累积 (Multi-stage, error accumulation)
- 需要相机内参 (Requires camera intrinsics)
- 弱纹理/重复纹理区域失败 (Weak/repetitive texture failures)
- 计算复杂度高 (High computational complexity)

In [ ]:
# 可视化传统SfM Pipeline的问题
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- 左侧: 传统SfM流程 ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('传统SfM Pipeline (COLMAP)\nTraditional SfM Pipeline', 
             fontsize=13, fontweight='bold', color='#D32F2F')

# 流程框
stages = [
    (1, 10.5, 8, 1.0, '输入: 图像序列\nInput: Image Sequence', '#E8EAF6', '#283593'),
    (1, 8.5, 3.5, 1.2, '特征提取\n(SIFT/SuperPoint)', '#FFEBEE', '#D32F2F'),
    (5.5, 8.5, 3.5, 1.2, '特征匹配\nFeature Matching', '#FFEBEE', '#D32F2F'),
    (1, 6.5, 3.5, 1.2, 'RANSAC\n几何验证', '#FFEBEE', '#D32F2F'),
    (5.5, 6.5, 3.5, 1.2, '增量式重建\nIncremental SfM', '#E3F2FD', '#1565C0'),
    (1, 4.5, 3.5, 1.2, 'Bundle\nAdjustment', '#E3F2FD', '#1565C0'),
    (5.5, 4.5, 3.5, 1.2, '稠密重建\n(MVS)', '#E8F5E9', '#2E7D32'),
    (1, 2.0, 8, 1.5, '输出: 相机位姿 + 点云\nOutput: Poses + Point Cloud', '#FFF3E0', '#E65100'),
]

for (x, y, w, h, text, color, edge_color) in stages:
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor=edge_color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=9, fontweight='bold')

# 问题标注
problems = [
    (0.5, 8.0, '离散!\nDiscrete', '#D32F2F'),
    (4.5, 7.5, '易错!\nError-prone', '#D32F2F'),
    (1.0, 5.5, '慢!\nSlow', '#D32F2F'),
]

for (x, y, text, color) in problems:
    ax.text(x, y, text, fontsize=8, color=color, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=color, alpha=0.8))

ax.text(5, 0.5, '多阶段 → 误差累积 → 需要相机内参', ha='center',
        fontsize=10, color='#D32F2F', style='italic', fontweight='bold')

# --- 右侧: 失败案例分析 ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('传统SfM的失败场景\nFailure Cases', 
             fontsize=13, fontweight='bold', color='#D32F2F')

cases = [
    (1, 9, 8, 2.0, '弱纹理区域\nWeak Texture', 
     '• 特征点检测失败\n• 无法找到足够的匹配点\n• 重建空洞', '#FFEBEE', '#D32F2F'),
    (1, 6, 8, 2.0, '重复纹理\nRepetitive Texture', 
     '• 特征匹配歧义\n• 错误匹配率高\n• RANSAC难以过滤', '#FFEBEE', '#D32F2F'),
    (1, 3, 8, 2.0, '大基线场景\nLarge Baseline', 
     '• 视角变化大\n• 特征描述失效\n• 几何验证困难', '#FFEBEE', '#D32F2F'),
]

for (x, y, w, h, title, desc, color, edge_color) in cases:
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor=edge_color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h - 0.3, title, ha='center', va='top',
            fontsize=11, fontweight='bold', color=edge_color)
    ax.text(x + 0.3, y + h/2, desc, ha='left', va='center',
            fontsize=9, linespacing=1.5)

plt.tight_layout()
plt.savefig('traditional_sfm_problems.png', dpi=150, bbox_inches='tight')
plt.show()

print("传统SfM的主要限制:")
print("  ✗ 多阶段pipeline，误差累积")
print("  ✗ 需要相机内参")
print("  ✗ 弱纹理/重复纹理区域失败")
print("  ✗ 计算复杂度高 (分钟到小时)")

## 2. DUSt3R Solution Overview

### 2.1 DUSt3R的解决方案

DUSt3R采用**端到端几何学习**范式，将传统多阶段pipeline替换为单次前馈神经网络:

```
DUSt3R Pipeline:
┌─────────────────────────────────────────────────────────────────┐
│  输入: 两张图像 I1, I2 (无需相机参数)                             │
│                    ↓                                             │
│  网络: ViT Encoder + Cross-Attention Decoder                      │
│                    ↓                                             │
│  输出: Pointmap P1, P2 (在统一坐标系下的3D点云)                    │
│                    ↓                                             │
│  后处理: 全局对齐 + 位姿估计                                       │
└─────────────────────────────────────────────────────────────────┘

优势:
- 端到端训练，统一优化
- 无需相机内参
- 对弱纹理更鲁棒
- 单次前馈，速度快
```

In [ ]:
# 可视化DUSt3R架构
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('DUSt3R Architecture: End-to-End Geometric Learning', 
             fontsize=16, fontweight='bold', color='#1565C0')

# 输入图像
img1_box = FancyBboxPatch((0.5, 8.5), 3, 2,
                          boxstyle="round,pad=0.15",
                          facecolor='#E8EAF6', edgecolor='#283593', linewidth=2.5)
ax.add_patch(img1_box)
ax.text(2, 9.5, 'Image 1\nI₁', ha='center', va='center',
        fontsize=12, fontweight='bold', color='#283593')

img2_box = FancyBboxPatch((12.5, 8.5), 3, 2,
                          boxstyle="round,pad=0.15",
                          facecolor='#E8EAF6', edgecolor='#283593', linewidth=2.5)
ax.add_patch(img2_box)
ax.text(14, 9.5, 'Image 2\nI₂', ha='center', va='center',
        fontsize=12, fontweight='bold', color='#283593')

# ViT Encoder
enc1 = FancyBboxPatch((0.5, 6), 3, 1.5,
                      boxstyle="round,pad=0.15",
                      facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2.5)
ax.add_patch(enc1)
ax.text(2, 6.75, 'ViT Encoder\n(Shared)', ha='center', va='center',
        fontsize=10, fontweight='bold', color='#1565C0')

enc2 = FancyBboxPatch((12.5, 6), 3, 1.5,
                      boxstyle="round,pad=0.15",
                      facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2.5)
ax.add_patch(enc2)
ax.text(14, 6.75, 'ViT Encoder\n(Shared)', ha='center', va='center',
        fontsize=10, fontweight='bold', color='#1565C0')

# 特征表示
ax.text(2, 5.2, 'F₁ ∈ R^(N×D)', ha='center', va='center',
        fontsize=9, style='italic', color='#666')
ax.text(14, 5.2, 'F₂ ∈ R^(N×D)', ha='center', va='center',
        fontsize=9, style='italic', color='#666')

# Cross-Attention Decoder
dec_box = FancyBboxPatch((4.5, 3.5), 7, 2.5,
                         boxstyle="round,pad=0.15",
                         facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3)
ax.add_patch(dec_box)
ax.text(8, 5.2, 'Cross-Attention Decoder', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#E65100')
ax.text(8, 4.5, '• Multi-layer Transformer', ha='center', va='center',
        fontsize=10, color='#666')
ax.text(8, 4.0, '• Self-attention (intra-view)', ha='center', va='center',
        fontsize=10, color='#666')
ax.text(8, 3.5, '• Cross-attention (inter-view)', ha='center', va='center',
        fontsize=10, color='#666')

# 箭头
ax.annotate('', xy=(2, 8.5), xytext=(2, 6),
            arrowprops=dict(arrowstyle='->', color='#283593', lw=2.5))
ax.annotate('', xy=(14, 8.5), xytext=(14, 6),
            arrowprops=dict(arrowstyle='->', color='#283593', lw=2.5))

# 从encoder到decoder
ax.annotate('', xy=(5, 4.75), xytext=(3.5, 5.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('', xy=(11, 4.75), xytext=(12.5, 5.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

# 输出Head
head1 = FancyBboxPatch((1, 1.5), 3, 1.5,
                       boxstyle="round,pad=0.15",
                       facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2.5)
ax.add_patch(head1)
ax.text(2.5, 2.5, 'Head 1', ha='center', va='center',
        fontsize=10, fontweight='bold', color='#2E7D32')
ax.text(2.5, 1.9, 'P₁ (H×W×3)\nConfidence₁', ha='center', va='center',
        fontsize=8, color='#666')

head2 = FancyBboxPatch((12, 1.5), 3, 1.5,
                       boxstyle="round,pad=0.15",
                       facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2.5)
ax.add_patch(head2)
ax.text(13.5, 2.5, 'Head 2', ha='center', va='center',
        fontsize=10, fontweight='bold', color='#2E7D32')
ax.text(13.5, 1.9, 'P₂ (H×W×3)\nConfidence₂', ha='center', va='center',
        fontsize=8, color='#666')

# 从decoder到head
ax.annotate('', xy=(2.5, 3), xytext=(6, 3.5),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))
ax.annotate('', xy=(13.5, 3), xytext=(10, 3.5),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))

# 底部输出说明
output_box = FancyBboxPatch((3.5, 0), 9, 1.2,
                            boxstyle="round,pad=0.1",
                            facecolor='#FFF9C4', edgecolor='#F57F17', 
                            linewidth=2, linestyle='--')
ax.add_patch(output_box)
ax.text(8, 0.6, '输出: 两个Pointmap (统一坐标系) + 置信度图', 
        ha='center', va='center', fontsize=11, fontweight='bold', color='#F57F17')

plt.tight_layout()
plt.savefig('dust3r_architecture.png', dpi=150, bbox_inches='tight')
plt.show()

print("DUSt3R的关键优势:")
print("  ✓ 端到端训练，统一优化")
print("  ✓ 无需相机内参")
print("  ✓ 对弱纹理更鲁棒")
print("  ✓ 单次前馈，速度快")

## 3. Pointmap Representation (Core Innovation)

### 3.1 什么是Pointmap?

Pointmap是DUSt3R的核心创新，它将传统的深度图表示升级为直接预测3D坐标:

```
传统深度图: 每个像素存储深度值 Z(u,v)
Pointmap: 每个像素存储3D坐标 X(u,v) = [X, Y, Z]

对比:
┌──────────────────────────────────────────────────────────────┐
│  深度图 (Depth Map)            Pointmap                       │
│                                                              │
│  [z11 z12 z13 ...]            [[x11,y11,z11] [x12,y12,z12]...]│
│  [z21 z22 z23 ...]            [[x21,y21,z21] [x22,y22,z22]...]│
│  [...          ]              [...                          ]│
│                                                              │
│  - 需要相机内参反投影           - 直接是3D坐标                 │
│  - 相机坐标系                   - 统一坐标系                   │
│  - 尺度不确定                   - 绝对尺度 (训练数据决定)        │
└──────────────────────────────────────────────────────────────┘
```

### 3.2 DUSt3R的双Pointmap输出

对于图像对 (I1, I2)，DUSt3R输出两个Pointmap，**关键创新在于它们在同一坐标系下**:

- **Pointmap1 (P1)**: I1中每个像素的3D坐标，坐标系以I1相机中心为原点
- **Pointmap2 (P2)**: I2中每个像素的3D坐标，坐标系**同样以I1相机中心为原点**

这意味着两个Pointmap可以直接配准，无需额外的位姿估计！

In [ ]:
# 可视化Pointmap表示
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- 左侧: 传统深度图表示 ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('传统: 深度图 + 位姿\nTraditional: Depth + Pose', 
             fontsize=13, fontweight='bold', color='#D32F2F')

# 图像1
ax.add_patch(FancyBboxPatch((0.5, 9), 3.5, 2, boxstyle="round,pad=0.1",
                           facecolor='#E8EAF6', edgecolor='#283593', linewidth=2))
ax.text(2.25, 10, 'Image 1', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(2.25, 9.4, 'Depth Map Z₁', ha='center', va='center', fontsize=9, color='#666')

# 图像2
ax.add_patch(FancyBboxPatch((6, 9), 3.5, 2, boxstyle="round,pad=0.1",
                           facecolor='#E8EAF6', edgecolor='#283593', linewidth=2))
ax.text(7.75, 10, 'Image 2', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(7.75, 9.4, 'Depth Map Z₂', ha='center', va='center', fontsize=9, color='#666')

# 相机内参
ax.add_patch(FancyBboxPatch((0.5, 6.5), 3.5, 1.5, boxstyle="round,pad=0.1",
                           facecolor='#FFEBEE', edgecolor='#D32F2F', linewidth=2))
ax.text(2.25, 7.6, 'Intrinsics K', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(2.25, 7.0, 'fx, fy, cx, cy', ha='center', va='center', fontsize=9, color='#666')

ax.add_patch(FancyBboxPatch((6, 6.5), 3.5, 1.5, boxstyle="round,pad=0.1",
                           facecolor='#FFEBEE', edgecolor='#D32F2F', linewidth=2))
ax.text(7.75, 7.6, 'Intrinsics K', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(7.75, 7.0, 'fx, fy, cx, cy', ha='center', va='center', fontsize=9, color='#666')

# 位姿估计
ax.add_patch(FancyBboxPatch((3, 4), 4, 1.5, boxstyle="round,pad=0.1",
                           facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2))
ax.text(5, 5.1, 'Pose Estimation T₁₂', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(5, 4.5, '[R|t] ∈ SE(3)', ha='center', va='center', fontsize=9, color='#666')

# 箭头
ax.annotate('', xy=(2.25, 6.5), xytext=(2.25, 9),
            arrowprops=dict(arrowstyle='->', color='#283593', lw=2))
ax.annotate('', xy=(7.75, 6.5), xytext=(7.75, 9),
            arrowprops=dict(arrowstyle='->', color='#283593', lw=2))
ax.annotate('', xy=(3, 4.75), xytext=(2.25, 6.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('', xy=(7, 4.75), xytext=(7.75, 6.5),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

# 3D空间
ax.add_patch(FancyBboxPatch((2, 1.5), 6, 2, boxstyle="round,pad=0.1",
                           facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))
ax.text(5, 2.8, '3D Point Cloud', ha='center', va='center', fontsize=11, fontweight='bold')
ax.text(5, 2.2, 'P1 = K⁻¹ @ Z1 @ pixels\nP2 = T₁₂ @ K⁻¹ @ Z2 @ pixels', 
        ha='center', va='center', fontsize=8, color='#666')

ax.annotate('', xy=(5, 3.5), xytext=(5, 4),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# 问题标注
ax.text(5, 0.3, '需要: 相机内参 + 位姿估计', ha='center', fontsize=10,
        color='#D32F2F', fontweight='bold')

# --- 中间: DUSt3R Pointmap表示 ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('DUSt3R: Pointmap\nDUSt3R: Pointmap', 
             fontsize=13, fontweight='bold', color='#1565C0')

# 图像1
ax.add_patch(FancyBboxPatch((0.5, 9), 3.5, 2, boxstyle="round,pad=0.1",
                           facecolor='#E8EAF6', edgecolor='#283593', linewidth=2))
ax.text(2.25, 10, 'Image 1', ha='center', va='center', fontsize=11, fontweight='bold')

# 图像2
ax.add_patch(FancyBboxPatch((6, 9), 3.5, 2, boxstyle="round,pad=0.1",
                           facecolor='#E8EAF6', edgecolor='#283593', linewidth=2))
ax.text(7.75, 10, 'Image 2', ha='center', va='center', fontsize=11, fontweight='bold')

# DUSt3R网络
ax.add_patch(FancyBboxPatch((1.5, 5.5), 7, 2.5, boxstyle="round,pad=0.15",
                           facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3))
ax.text(5, 7.2, 'DUSt3R Network', ha='center', va='center', fontsize=13, fontweight='bold', color='#E65100')
ax.text(5, 6.5, 'ViT + Cross-Attention', ha='center', va='center', fontsize=10, color='#666')

# 箭头
ax.annotate('', xy=(2.25, 8), xytext=(2.25, 9),
            arrowprops=dict(arrowstyle='->', color='#283593', lw=2))
ax.annotate('', xy=(7.75, 8), xytext=(7.75, 9),
            arrowprops=dict(arrowstyle='->', color='#283593', lw=2))
ax.annotate('', xy=(3.5, 6.75), xytext=(2.25, 8),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('', xy=(6.5, 6.75), xytext=(7.75, 8),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))

# Pointmap输出
ax.add_patch(FancyBboxPatch((0.5, 3), 3.5, 2, boxstyle="round,pad=0.1",
                           facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))
ax.text(2.25, 4.3, 'Pointmap P₁', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(2.25, 3.7, '(H×W×3)\nCoord: Camera 1', ha='center', va='center', fontsize=8, color='#666')

ax.add_patch(FancyBboxPatch((6, 3), 3.5, 2, boxstyle="round,pad=0.1",
                           facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))
ax.text(7.75, 4.3, 'Pointmap P₂', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(7.75, 3.7, '(H×W×3)\nCoord: Camera 1', ha='center', va='center', fontsize=8, color='#666')

ax.annotate('', xy=(2.25, 5), xytext=(4, 5.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.annotate('', xy=(7.75, 5), xytext=(6, 5.5),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# 关键标注
ax.add_patch(FancyBboxPatch((1.5, 1), 7, 1.5, boxstyle="round,pad=0.1",
                           facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2, linestyle='--'))
ax.text(5, 2.0, '同一坐标系! (Same coordinate system)', ha='center', va='center', 
        fontsize=11, fontweight='bold', color='#F57F17')
ax.text(5, 1.4, '无需相机内参，直接配准', ha='center', va='center', fontsize=9, color='#666')

# --- 右侧: 数学对比 ---
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('表示对比\nRepresentation Comparison', 
             fontsize=13, fontweight='bold')

# 深度图矩阵
ax.add_patch(FancyBboxPatch((0.5, 8.5), 4, 3, boxstyle="round,pad=0.1",
                           facecolor='#FFEBEE', edgecolor='#D32F2F', linewidth=2))
ax.text(2.5, 11.2, '深度图 Depth', ha='center', va='center', fontsize=11, fontweight='bold', color='#D32F2F')
ax.text(2.5, 10.0, '[z₁₁ z₁₂ z₁₃ ...]', ha='center', va='center', fontsize=9, family='monospace')
ax.text(2.5, 9.5, '[z₂₁ z₂₂ z₂₃ ...]', ha='center', va='center', fontsize=9, family='monospace')
ax.text(2.5, 9.0, '[...          ]', ha='center', va='center', fontsize=9, family='monospace')
ax.text(2.5, 8.0, '维度: H×W', ha='center', va='center', fontsize=9, color='#666')

# Pointmap矩阵
ax.add_patch(FancyBboxPatch((5.5, 8.5), 4, 3, boxstyle="round,pad=0.1",
                           facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2))
ax.text(7.5, 11.2, 'Pointmap', ha='center', va='center', fontsize=11, fontweight='bold', color='#2E7D32')
ax.text(7.5, 10.0, '[[x₁₁,y₁₁,z₁₁]...]', ha='center', va='center', fontsize=9, family='monospace')
ax.text(7.5, 9.5, '[[x₂₁,y₂₁,z₂₁]...]', ha='center', va='center', fontsize=9, family='monospace')
ax.text(7.5, 9.0, '[...             ]', ha='center', va='center', fontsize=9, family='monospace')
ax.text(7.5, 8.0, '维度: H×W×3', ha='center', va='center', fontsize=9, color='#666')

# 优势对比
ax.add_patch(FancyBboxPatch((0.5, 4), 9, 3.5, boxstyle="round,pad=0.1",
                           facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2))
ax.text(5, 7.0, 'Pointmap优势 Advantages', ha='center', va='center', 
        fontsize=12, fontweight='bold', color='#1565C0')

advantages = [
    '✓ 直接是3D坐标 (Direct 3D coordinates)',
    '✓ 统一坐标系 (Unified coordinate system)',
    '✓ 无需相机内参 (No intrinsics needed)',
    '✓ 天然配准 (Natural registration)',
]
for i, adv in enumerate(advantages):
    ax.text(1, 6.2 - i*0.5, adv, ha='left', va='center', fontsize=10, color='#333')

# 坐标系说明
ax.add_patch(FancyBboxPatch((0.5, 0.5), 9, 3, boxstyle="round,pad=0.1",
                           facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2))
ax.text(5, 3.0, '坐标系对比 Coordinate Systems', ha='center', va='center', 
        fontsize=11, fontweight='bold', color='#E65100')

ax.text(1, 2.3, '深度图: 各自相机坐标系', ha='left', va='center', fontsize=9, color='#333')
ax.text(1, 1.8, 'Pointmap: 统一以Camera 1为原点', ha='left', va='center', fontsize=9, color='#333')
ax.text(1, 1.2, '关键: P₁和P₂可直接比较、配准!', ha='left', va='center', 
        fontsize=9, fontweight='bold', color='#E65100')

plt.tight_layout()
plt.savefig('pointmap_representation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Pointmap的核心优势:")
print("  ✓ 直接预测3D坐标，无需反投影")
print("  ✓ 统一坐标系，天然配准")
print("  ✓ 无需相机内参")
print("  ✓ 端到端可微分")

## 4. DUSt3R vs COLMAP Comparison

### 4.1 性能对比

在DTU数据集上的对比:

| 方法 | 推理时间 | 精度 (Chamfer Distance) | 需要内参 |
|------|---------|----------------------|---------|
| COLMAP | ~25s | 0.89mm | 是 |
| DUSt3R | ~0.8s | 0.92mm | 否 |

**结论**:
- DUSt3R快约30倍
- 精度相当
- 无需相机内参

In [ ]:
# DUSt3R vs COLMAP对比
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# --- 左侧: 性能对比图 ---
ax = axes[0]

methods = ['COLMAP', 'DUSt3R']
time_seconds = [25, 0.8]
accuracy = [0.89, 0.92]  # Chamfer distance (lower is better)

x = np.arange(len(methods))
width = 0.35

# 时间对比
bars1 = ax.bar(x - width/2, time_seconds, width, label='时间 Time (s)', 
               color=['#D32F2F', '#1565C0'], alpha=0.8)
ax.set_ylabel('时间 Time (seconds)', color='#D32F2F', fontsize=12, fontweight='bold')
ax.set_yscale('log')
ax.tick_params(axis='y', labelcolor='#D32F2F')

# 精度对比 (右侧Y轴)
ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, accuracy, width, label='精度 Accuracy (mm)', 
                color=['#D32F2F', '#1565C0'], alpha=0.4, hatch='///')
ax2.set_ylabel('精度 Accuracy (mm) ↓', color='#1565C0', fontsize=12, fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#1565C0')
ax2.set_ylim(0, 1.5)

ax.set_xticks(x)
ax.set_xticklabels(methods, fontsize=12, fontweight='bold')
ax.set_title('DUSt3R vs COLMAP: DTU Dataset', fontsize=13, fontweight='bold')

# 添加数值标签
for bar in bars1:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')

for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}mm', ha='center', va='bottom', fontsize=10, fontweight='bold')

# 速度提升标注
speedup = time_seconds[0] / time_seconds[1]
ax.text(0.5, 15, f'速度提升\n{speedup:.0f}× Faster!', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#1565C0',
        bbox=dict(boxstyle='round', facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2))

# --- 右侧: 综合对比表 ---
ax = axes[1]
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title('详细对比 Detailed Comparison', fontsize=13, fontweight='bold')

comparison = [
    ('维度', 'COLMAP', 'DUSt3R'),
    ('方法类型', '传统多阶段', '端到端神经网络'),
    ('需要内参', '✗ 必须', '✓ 不需要'),
    ('特征提取', 'SIFT/SuperPoint', 'ViT Encoder'),
    ('匹配方法', '最近邻+RANSAC', 'Cross-Attention'),
    ('重建方式', '增量式+BA', '直接Pointmap'),
    ('弱纹理', '✗ 易失败', '✓ 较鲁棒'),
    ('可微分', '✗ 否', '✓ 是'),
    ('端到端训练', '✗ 否', '✓ 是'),
    ('速度 (DTU)', '25s', '0.8s'),
    ('精度', '0.89mm', '0.92mm'),
]

y_start = 11
for i, (dim, colmap, dust3r) in enumerate(comparison):
    if i == 0:
        # 表头
        ax.add_patch(FancyBboxPatch((0.2, y_start - 0.3), 2.5, 0.6, 
                                   boxstyle="round,pad=0.05", facecolor='#333', edgecolor='black'))
        ax.add_patch(FancyBboxPatch((2.9, y_start - 0.3), 3, 0.6, 
                                   boxstyle="round,pad=0.05", facecolor='#D32F2F', edgecolor='black'))
        ax.add_patch(FancyBboxPatch((6.2, y_start - 0.3), 3, 0.6, 
                                   boxstyle="round,pad=0.05", facecolor='#1565C0', edgecolor='black'))
        ax.text(1.45, y_start, dim, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
        ax.text(4.4, y_start, colmap, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
        ax.text(7.7, y_start, dust3r, ha='center', va='center', fontsize=11, fontweight='bold', color='white')
    else:
        # 数据行
        bg_color = '#F5F5F5' if i % 2 == 1 else 'white'
        ax.add_patch(FancyBboxPatch((0.2, y_start - i*0.9 - 0.3), 2.5, 0.6, 
                                   boxstyle="round,pad=0.05", facecolor=bg_color, edgecolor='gray', linewidth=0.5))
        ax.add_patch(FancyBboxPatch((2.9, y_start - i*0.9 - 0.3), 3, 0.6, 
                                   boxstyle="round,pad=0.05", facecolor='#FFEBEE', edgecolor='gray', linewidth=0.5))
        ax.add_patch(FancyBboxPatch((6.2, y_start - i*0.9 - 0.3), 3, 0.6, 
                                   boxstyle="round,pad=0.05", facecolor='#E3F2FD', edgecolor='gray', linewidth=0.5))
        ax.text(1.45, y_start - i*0.9, dim, ha='center', va='center', fontsize=10, fontweight='bold')
        ax.text(4.4, y_start - i*0.9, colmap, ha='center', va='center', fontsize=10)
        ax.text(7.7, y_start - i*0.9, dust3r, ha='center', va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('dust3r_vs_colmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("DUSt3R vs COLMAP总结:")
print("  ✓ 速度快30倍")
print("  ✓ 精度相当")
print("  ✓ 无需相机内参")
print("  ✓ 端到端可训练")
print("  ✗ 对重复纹理仍可能失败")

## 5. Evolution: Pairwise → Multi-view

### 5.1 从DUSt3R到Fast3R到VGGT

DUSt3R开创了pairwise几何学习范式，但其O(N²)复杂度限制了多视图扩展。后续工作逐步解决了这个问题:

**DUSt3R (CVPR 2024)**: 基础pairwise方法
- O(N²) forward passes for N images
- 需要全局对齐

**Fast3R (2024)**: 优化的pairwise方法
- 更好的全局对齐算法
- 仍然O(N²)，但更快

**VGGT (2024)**: 全对全注意力
- O(N) forward passes
- 单次前馈处理所有图像
- 输出位姿、相机、深度、Gaussians

In [ ]:
# 可视化演进时间线
fig, ax = plt.subplots(figsize=(16, 10))
ax.set_xlim(0, 16)
ax.set_ylim(0, 12)
ax.axis('off')
ax.set_title('Evolution: Pairwise → Multi-view Methods', fontsize=16, fontweight='bold')

# 时间线主干
timeline_y = 10
ax.plot([1, 15], [timeline_y, timeline_y], 'k-', linewidth=3, alpha=0.3)
ax.text(0.5, timeline_y, '2024', fontsize=10, ha='right', va='center', color='gray')
ax.text(15.5, timeline_y, 'Future', fontsize=10, ha='left', va='center', color='gray')

# 方法框
methods = [
    (0.5, 6, 3.5, 3, 'DUSt3R\n(CVPR 2024)', 
     '• Pairwise ViT\n• Cross-attention\n• O(N²) scaling\n• Pointmap output\n• Global alignment needed',
     '#E3F2FD', '#1565C0', 2.5),
    
    (4.5, 6.5, 3.5, 2.5, 'Fast3R\n(2024)',
     '• Optimized DUSt3R\n• Better alignment\n• Still O(N²)\n• Confidence-based',
     '#E8F5E9', '#2E7D32', 6.5),
    
    (8.5, 5.5, 3.5, 3.5, 'MASt3R\n(ECCV 2024)',
     '• Matching improvement\n• Feature-level matching\n• Better correspondences\n• Still pairwise\n• For large scenes',
     '#F3E5F5', '#6A1B9A', 10.5),
    
    (12, 4.5, 3.5, 4.5, 'VGGT\n(2024)',
     '• All-to-all attention\n• O(N) scaling\n• 4 unified outputs\n• Pose + Camera\n• Depth + Gaussians\n• SLAM-ready',
     '#FFF3E0', '#E65100', 14),
]

for (x, y, w, h, name, details, color, edge_color, year_x) in methods:
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.15",
                         facecolor=color, edgecolor=edge_color, linewidth=2.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h - 0.4, name, ha='center', va='top',
            fontsize=11, fontweight='bold', color=edge_color)
    ax.text(x + w/2, y + 0.4, details, ha='center', va='bottom',
            fontsize=8, color='black', linespacing=1.5)
    
    # 时间线标记
    ax.plot([year_x, year_x], [timeline_y-0.2, timeline_y+0.2], 'o', 
            markersize=10, color=edge_color, markeredgecolor='white', markeredgewidth=2)
    ax.plot([year_x, x + w/2], [timeline_y, y + h], '--', 
            color=edge_color, linewidth=1.5, alpha=0.5)

# 演进箭头
evolution_y = 5
ax.annotate('', xy=(4.5, evolution_y), xytext=(4, evolution_y),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.annotate('', xy=(8.5, evolution_y), xytext=(8, evolution_y),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.annotate('', xy=(12, evolution_y+1), xytext=(11.5, evolution_y+1),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))

# 关键创新
innovations = [
    (2.5, 4.2, '开创性\nPioneering', '#1565C0'),
    (6.5, 4.7, '优化对齐\nBetter Align', '#2E7D32'),
    (10.5, 3.7, '匹配改进\nMatching', '#6A1B9A'),
    (14, 2.7, '全对全\nAll-to-all', '#E65100'),
]

for (x, y, text, color) in innovations:
    ax.text(x, y, text, ha='center', va='center', fontsize=9,
            color=color, fontweight='bold', style='italic',
            bbox=dict(boxstyle='round', facecolor='white', edgecolor=color, linewidth=1.5))

# 复杂度对比在底部
scaling_y = 1.5
ax.text(8, scaling_y + 0.8, '复杂度演进 Complexity Evolution:', ha='center', fontsize=11, fontweight='bold')
scaling_boxes = [
    (1.5, scaling_y, 'DUSt3R\nO(N²)', '#BBDEFB', '#1565C0'),
    (5.5, scaling_y, 'Fast3R\nO(N²)', '#C8E6C9', '#2E7D32'),
    (9.5, scaling_y, 'MASt3R\nO(N²)', '#E1BEE7', '#6A1B9A'),
    (13.5, scaling_y, 'VGGT\nO(N)', '#FFCC80', '#E65100'),
]

for (x, y, text, color, edge_color) in scaling_boxes:
    box = FancyBboxPatch((x-0.8, y-0.3), 1.6, 0.6,
                         boxstyle="round,pad=0.08",
                         facecolor=color, edgecolor=edge_color, linewidth=2)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('evolution_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("演进关键点:")
print("  1. DUSt3R → Fast3R: 更好的全局对齐算法")
print("  2. DUSt3R → MASt3R: 匹配层面的改进")
print("  3. 所有pairwise方法 → VGGT: O(N²) → O(N)的范式转变")

# 可视化复杂度增长
fig, ax = plt.subplots(figsize=(10, 6))

N = np.arange(2, 51)
pairwise = N * (N - 1) / 2  # O(N²)
all_to_all = N  # O(N)

ax.plot(N, pairwise, 'b-', linewidth=2.5, label='Pairwise (DUSt3R/Fast3R) O(N²)', marker='o', markevery=5)
ax.plot(N, all_to_all, 'r-', linewidth=2.5, label='All-to-all (VGGT) O(N)', marker='s', markevery=5)

ax.fill_between(N, pairwise, all_to_all, alpha=0.2, color='red')

ax.set_xlabel('Number of Images (N)', fontsize=12, fontweight='bold')
ax.set_ylabel('Forward Passes Required', fontsize=12, fontweight='bold')
ax.set_title('Computational Complexity: Pairwise vs All-to-all', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

# 标注
ax.annotate('50 images → 1,225 passes', xy=(50, pairwise[-1]), xytext=(35, 800),
            fontsize=10, color='#1565C0', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('50 images → 50 passes', xy=(50, all_to_all[-1]), xytext=(35, 150),
            fontsize=10, color='#E65100', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2))

plt.tight_layout()
plt.savefig('complexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n对于50张图像:")
print(f"  Pairwise方法: {int(50*49/2):,} 次前向传播")
print(f"  All-to-all方法: {50} 次前向传播")
print(f"  → 24倍效率提升!")

## 6. Connection to SLAM

### 6.1 传统SLAM前端 vs DUSt3R

传统SLAM前端采用多步骤离散处理:

```
传统SLAM前端 (如ORB-SLAM3):
┌─────────────┐   ┌─────────────┐   ┌─────────────┐   ┌──────────┐
│ 特征提取    │──►│ 特征匹配    │──►│ RANSAC验证  │──►│ 位姿估计 │
│ (SIFT/ORB)  │   │ (最近邻)    │   │ (几何约束)  │   │ (EPnP)   │
└─────────────┘   └─────────────┘   └─────────────┘   └──────────┘
   离散              可能错误            鲁棒但慢           需要内参

DUSt3R:
┌─────────────────────────────────────────────────────────────────┐
│                    端到端网络 (单次前馈)                          │
│  输入: 图像对 ──► ViT Encoder ──► Cross-Attention ──► Pointmap  │
│  输出: 直接是3D点云，无需相机内参                                 │
│  置信度: 自动识别可靠/不可靠区域                                  │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
# 可视化SLAM前端对比
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- 左侧: 传统SLAM前端 ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title('传统SLAM前端 Traditional SLAM Frontend', fontsize=13, fontweight='bold', color='#D32F2F')
ax.axis('off')

slam_boxes = [
    (1, 10, 8, 1.0, '输入: 图像帧\nInput: Video Frames', '#E8EAF6', '#283593'),
    (1, 8.2, 3.5, 1.3, '特征提取\nFeature Extraction', '#FFEBEE', '#D32F2F'),
    (5.5, 8.2, 3.5, 1.3, '特征匹配\nFeature Matching', '#FFEBEE', '#D32F2F'),
    (1, 6.2, 3.5, 1.3, 'RANSAC\nOutlier Rejection', '#FFEBEE', '#D32F2F'),
    (5.5, 6.2, 3.5, 1.3, '位姿估计\nPose Estimation', '#E3F2FD', '#1565C0'),
    (1, 4.0, 8, 1.5, '输出: 相机位姿\nOutput: Camera Pose', '#FFF3E0', '#E65100'),
]

for (x, y, w, h, text, color, edge_color) in slam_boxes:
    box = FancyBboxPatch((x, y), w, h,
                         boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor=edge_color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=10, fontweight='bold')

# 箭头
arrows = [
    ((3.25, 9), (3.25, 9.5)),
    ((7.25, 9), (7.25, 9.5)),
    ((3.25, 6.9), (3.25, 8.2)),
    ((7.25, 6.9), (7.25, 8.2)),
    ((3.25, 4.9), (3.25, 6.2)),
    ((5, 5.5), (5, 6.2)),
]

for start, end in arrows:
    ax.annotate('', xy=end, xytext=start,
                arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# 问题标注
ax.text(5, 2.8, '问题:', ha='center', fontsize=11, fontweight='bold', color='#D32F2F')
problems = [
    '• 多阶段，误差累积',
    '• 需要相机内参',
    '• RANSAC非可微分',
    '• 无法端到端训练',
]
for i, prob in enumerate(problems):
    ax.text(2, 2.2 - i*0.4, prob, ha='left', fontsize=9, color='#333')

# --- 右侧: DUSt3R ---
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 12)
ax.set_title('DUSt3R: 端到端几何学习\nDUSt3R: End-to-End Learning', fontsize=13, fontweight='bold', color='#2E7D32')
ax.axis('off')

# DUSt3R流程
ax.add_patch(FancyBboxPatch((1, 9.5), 8, 1.2, boxstyle="round,pad=0.1",
                           facecolor='#E8EAF6', edgecolor='#283593', linewidth=2))
ax.text(5, 10.1, '输入: 图像对 (无需相机参数)', ha='center', va='center', fontsize=11, fontweight='bold')

ax.add_patch(FancyBboxPatch((2, 6.5), 6, 2.5, boxstyle="round,pad=0.15",
                           facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3))
ax.text(5, 8.2, 'DUSt3R Network', ha='center', va='center', fontsize=13, fontweight='bold', color='#E65100')
ax.text(5, 7.4, 'ViT + Cross-Attention', ha='center', va='center', fontsize=10, color='#666')

# 箭头
ax.annotate('', xy=(5, 9.0), xytext=(5, 9.5),
            arrowprops=dict(arrowstyle='->', color='#283593', lw=2.5))

# 输出
outputs = [
    (1, 4.2, 3.5, 1.8, 'Pointmap P₁', '#E8F5E9', '#2E7D32'),
    (5.5, 4.2, 3.5, 1.8, 'Pointmap P₂', '#E8F5E9', '#2E7D32'),
]

for (x, y, w, h, text, color, edge_color) in outputs:
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.1",
                         facecolor=color, edgecolor=edge_color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h - 0.3, text, ha='center', va='top', fontsize=10, fontweight='bold')
    ax.text(x + w/2, y + 0.4, '(H×W×3)', ha='center', va='center', fontsize=8, color='#666')
    ax.annotate('', xy=(x + w/2, 5.8), xytext=(5, 6.5),
                arrowprops=dict(arrowstyle='->', color=edge_color, lw=2))

# 从Pointmap推导位姿
ax.add_patch(FancyBboxPatch((3, 2.2), 4, 1.5, boxstyle="round,pad=0.1",
                           facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2))
ax.text(5, 3.2, '位姿估计', ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(5, 2.6, 'Procrustes / SVD', ha='center', va='center', fontsize=9, color='#666')

ax.annotate('', xy=(3.5, 3.2), xytext=(2.75, 4.2),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.annotate('', xy=(6.5, 3.2), xytext=(7.25, 4.2),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))

# 优势标注
ax.text(5, 1.2, '优势 Advantages:', ha='center', fontsize=11, fontweight='bold', color='#2E7D32')
advantages = [
    '✓ 端到端可微分',
    '✓ 无需相机内参',
    '✓ 单次前馈',
    '✓ 可端到端训练',
]
for i, adv in enumerate(advantages):
    ax.text(2.5, 0.7 - i*0.35, adv, ha='left', fontsize=9, color='#333')

plt.tight_layout()
plt.savefig('slam_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("DUSt3R vs 传统SLAM前端:")
print("\n传统SLAM:")
print("  ✗ 多阶段pipeline，误差累积")
print("  ✗ 需要相机内参")
print("  ✗ RANSAC非可微分")
print("\nDUSt3R:")
print("  ✓ 端到端可微分")
print("  ✓ 无需相机内参")
print("  ✓ 置信度图自动识别可靠区域")
print("  ✓ 可端到端训练")

## 7. Phase 3 Learning Roadmap

### 7.1 学习路径

Phase 3将深入理解DUSt3R从理论到实践。

In [ ]:
# Phase 3学习路线图
roadmap = {
    '#': ['00', '01', '02', '03', '04', '05', '06', '07', '08'],
    'Topic': [
        'Overview & Motivation',
        'Pointmap Deep Dive',
        'ViT Encoder Architecture',
        'Cross-Attention Decoder',
        'Training & Loss Design',
        'Code Walkthrough',
        'Inference & Visualization',
        'Downstream Tasks (Pose, Depth)',
        'Extensions (Fast3R, MASt3R)',
    ],
    'Key Concepts': [
        'Why DUSt3R, Pointmap intro, COLMAP comparison',
        'Pointmap representation, coordinate systems',
        'ViT architecture, patch embedding, pretraining',
        'Cross-attention, CroCo pretraining',
        'Regression loss, confidence loss, training strategy',
        'Official code analysis, model structure',
        'Running inference, visualization',
        'Pose estimation, depth prediction, global alignment',
        'Fast3R improvements, MASt3R matching',
    ],
    'Time': [
        '50 min',
        '55 min',
        '60 min',
        '65 min',
        '70 min',
        '80 min',
        '60 min',
        '55 min',
        '45 min',
    ],
}

print("=" * 120)
print(f"{'#':3s} | {'Topic':30s} | {'Key Concepts':55s} | {'Time':10s}")
print("=" * 120)
for i in range(len(roadmap['#'])):
    num = roadmap['#'][i]
    topic = roadmap['Topic'][i]
    concepts = roadmap['Key Concepts'][i]
    time = roadmap['Time'][i]
    print(f"{num:3s} | {topic:30s} | {concepts:55s} | {time:10s}")
print("=" * 120)
print(f"\n总预计时间: ~9小时")
print(f"\n前置知识:")
print(f"  - Phase 1: 3DGS基础")
print(f"  - Phase 2: SLAM与相机几何")
print(f"  - 基础深度学习知识")

# 可视化学习周计划
fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Phase 3 Learning Schedule (3 Weeks)', fontsize=16, fontweight='bold')

weeks = [
    (1, 8, 'Week 1: 理论基础',
     ['Day 1-2: DUSt3R核心思想', 'Day 3-4: Pointmap表示', 'Day 5-7: 网络架构']),
    (5.5, 8, 'Week 2: 代码实践',
     ['Day 8-10: 官方代码走读', 'Day 11-12: 推理与可视化', 'Day 13-14: 与COLMAP对比']),
    (10, 8, 'Week 3: 进阶扩展',
     ['Day 15-17: 下游任务', 'Day 18-21: Fast3R/MASt3R']),
]

colors = ['#E3F2FD', '#E8F5E9', '#FFF3E0']
edge_colors = ['#1565C0', '#2E7D32', '#E65100']

for i, (x, y, title, tasks) in enumerate(weeks):
    # 周标题
    ax.add_patch(FancyBboxPatch((x-0.3, y+0.5), 3.6, 0.8,
                               boxstyle="round,pad=0.1",
                               facecolor=colors[i], edgecolor=edge_colors[i], linewidth=3))
    ax.text(x+1.5, y+0.9, title, ha='center', va='center',
            fontsize=12, fontweight='bold', color=edge_colors[i])
    
    # 任务列表
    for j, task in enumerate(tasks):
        ax.add_patch(FancyBboxPatch((x, y-1.2-j*1.2), 3, 1,
                                   boxstyle="round,pad=0.1",
                                   facecolor='white', edgecolor=edge_colors[i], linewidth=1.5))
        ax.text(x+1.5, y-0.7-j*1.2, task, ha='center', va='center', fontsize=9)

# 底部里程碑
ax.add_patch(FancyBboxPatch((2, 1), 10, 1.2,
                           boxstyle="round,pad=0.15",
                           facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2, linestyle='--'))
ax.text(7, 1.8, 'Phase 3 完成检查点', ha='center', va='center', fontsize=12, fontweight='bold', color='#F57F17')
ax.text(7, 1.3, '✓ 理解DUSt3R与传统SfM区别  ✓ 运行DUSt3R推理  ✓ 理解Pointmap表示', 
        ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('learning_roadmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary

### 8.1 关键要点总结

In [ ]:
# 总结框
summary = """
╔════════════════════════════════════════════════════════════════════════════════╗
║                   Phase 3 Overview - Summary                                  ║
╠════════════════════════════════════════════════════════════════════════════════╣
║                                                                                ║
║  1. WHY DUSt3R? 传统SfM的局限                                                    ║
║     - 多阶段pipeline，误差累积                                                  ║
║     - 需要相机内参                                                             ║
║     - 弱纹理/重复纹理区域失败                                                   ║
║     - 计算复杂度高                                                             ║
║                                                                                ║
║  2. DUSt3R解决方案: 端到端几何学习                                               ║
║     - ViT Encoder + Cross-Attention Decoder                                    ║
║     - 单次前馈，速度快                                                          ║
║     - 无需相机内参                                                             ║
║     - 端到端可训练                                                             ║
║                                                                                ║
║  3. POINTMAP表示 (核心创新)                                                     ║
║     - 直接预测3D坐标 [X,Y,Z] 而非深度Z                                          ║
║     - 统一坐标系，天然配准                                                      ║
║     - 两个Pointmap在同一原点下                                                 ║
║     - 无需相机内参反投影                                                        ║
║                                                                                ║
║  4. DUSt3R vs COLMAP                                                           ║
║     - 速度快30倍 (0.8s vs 25s)                                                 ║
║     - 精度相当 (0.92mm vs 0.89mm)                                              ║
║     - 无需相机内参                                                             ║
║                                                                                ║
║  5. 演进: Pairwise → Multi-view                                                 ║
║     - DUSt3R: O(N²), 需要全局对齐                                              ║
║     - Fast3R: 更好的对齐，仍然O(N²)                                            ║
║     - VGGT: O(N), 单次前馈处理所有图像                                          ║
║                                                                                ║
║  6. 与SLAM的联系                                                                ║
║     - 传统SLAM: 多阶段离散处理                                                  ║
║     - DUSt3R: 端到端神经网络替代                                                ║
║     - 可微分，可端到端训练                                                      ║
║                                                                                ║
║  7. NEXT STEPS                                                                 ║
║     - Notebook 01: Pointmap深入理解                                            ║
║     - Notebook 02-03: 网络架构 (ViT + Cross-Attention)                         ║
║     - Notebook 04: 训练与损失设计                                              ║
║     - Notebook 05-06: 代码实践与推理                                           ║
║     - Notebook 07-08: 下游任务与扩展                                           ║
║                                                                                ║
╚════════════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

# 生成一个简洁的要点图
fig, ax = plt.subplots(figsize=(12, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')

# 中央概念
ax.add_patch(FancyBboxPatch((3.5, 4), 3, 2,
                           boxstyle="round,pad=0.2",
                           facecolor='#FFF3E0', edgecolor='#E65100', linewidth=3))
ax.text(5, 5.5, 'DUSt3R', ha='center', va='center', fontsize=20, fontweight='bold', color='#E65100')
ax.text(5, 4.7, 'End-to-End\nGeometric Learning', ha='center', va='center', fontsize=11, color='#666')

# 四个关键要素
elements = [
    (1, 7, 'Pointmap\nRepresentation', '#2E7D32'),
    (7, 7, 'No Camera\nIntrinsics', '#1565C0'),
    (1, 2, 'Single\nForward Pass', '#6A1B9A'),
    (7, 2, 'Differentiable\n& Trainable', '#D32F2F'),
]

for (x, y, text, color) in elements:
    ax.add_patch(FancyBboxPatch((x, y), 2, 1.2,
                               boxstyle="round,pad=0.15",
                               facecolor='white', edgecolor=color, linewidth=2))
    ax.text(x+1, y+0.6, text, ha='center', va='center', fontsize=10, fontweight='bold', color=color)

# 箭头
ax.annotate('', xy=(3.5, 5), xytext=(3, 6.8),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2))
ax.annotate('', xy=(6.5, 5), xytext=(7, 6.8),
            arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
ax.annotate('', xy=(3.5, 4.8), xytext=(3, 3.2),
            arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=2))
ax.annotate('', xy=(6.5, 4.8), xytext=(7, 3.2),
            arrowprops=dict(arrowstyle='->', color='#D32F2F', lw=2))

ax.set_title('DUSt3R: Four Key Innovations', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('summary_innovations.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. What's Next?

在下一个notebook中，我们将深入理解**Pointmap表示**:

**[01_pointmap_deep_dive.ipynb](./01_pointmap_deep_dive.ipynb)** - Pointmap深度解析

我们将学习:
- Pointmap的数学定义与性质
- 坐标系转换
- 从Pointmap推导位姿 (Procrustes分析)
- 实际代码示例
- 可视化技巧

---

## References

1. **DUSt3R Paper**: "DUSt3R: Geometric 3D Vision Made Easy" CVPR 2024
   - [arXiv:2312.14132](https://arxiv.org/abs/2312.14132)

2. **CroCo**: "CroCo: Self-Supervised Pretraining for 3D Vision Tasks"
   - [arXiv:2210.02567](https://arxiv.org/abs/2210.02567)

3. **MASt3R**: "Grounding Image Matching in 3D with MASt3R" ECCV 2024

4. **COLMAP**: https://colmap.github.io/

5. **Fast3R**: "Fast3R: Towards Fast and Accurate 3D Reconstruction"

---

## Installation (后续Notebook)

```bash
# DUSt3R
git clone https://github.com/naver/dust3r.git
cd dust3r

conda create -n dust3r python=3.11
conda activate dust3r

pip install torch==2.2.0 torchvision --index-url https://download.pytorch.org/whl/cu121
pip install -r requirements.txt

# 下载预训练模型
mkdir -p checkpoints
# 从Hugging Face下载 DUSt3R_ViTLarge_BaseDec512Dpt.pth
```

我们将在Notebook 06中详细介绍环境搭建和代码实践!